# Stargazer Engineer-Agent Experiment

This notebook runs one real Stargazer engineer-agent experiment. It shows the provider check, public task preview, planner criteria, engineer/critic/executor loop, final JSON artifact, and whether the artifact is ready for the separate evaluation notebook.

The hidden Stargazer benchmark is not used here. The critic accepts or rejects the engineer output only by public contract, execution result, and JSON schema readiness.


## 1. Setup

This cell defines paths, loads the local provider env, and prepares a notebook-local output folder. The API key is checked but not printed.


In [1]:
from pathlib import Path
import ast
import json
import math
import os
import shutil
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display
from openai import OpenAI


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "traj_eval").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the traj-eval checkout.")


def load_env_file(path: Path) -> dict[str, str]:
    loaded = {}
    for line in path.read_text(encoding="utf-8-sig").splitlines():
        text = line.strip().lstrip("\ufeff")
        if not text or text.startswith("#") or "=" not in text:
            continue
        key, value = text.split("=", 1)
        loaded[key.strip()] = value.strip().strip('"').strip("'")
        os.environ[key.strip()] = loaded[key.strip()]
    return loaded


def write_json(path: Path, payload) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")


def run_command(args, cwd: Path, timeout: int = 600) -> dict:
    start = time.time()
    proc = subprocess.run(args, cwd=str(cwd), text=True, capture_output=True, timeout=timeout)
    return {
        "args": [str(item) for item in args],
        "returncode": proc.returncode,
        "elapsed_seconds": round(time.time() - start, 3),
        "stdout": proc.stdout,
        "stderr": proc.stderr,
    }


repo_root = find_repo_root(Path.cwd().resolve())
notebook_dir = repo_root / "notebooks" / "qwen_saeed_stargazer_real1"
out_dir = notebook_dir / "outputs" / "qwen_saeed_agent_stargazer"
work_dir = out_dir / "agent_workflow"
source_dir = repo_root / "src"
support_dir = notebook_dir / "support"
obs_path = notebook_dir / "tasks" / "stargazer_real_real_001_minimal" / "stargazer_observations.json"

for required_path in [source_dir, support_dir / "stargazer", obs_path]:
    assert required_path.exists(), f"Missing required path: {required_path}"
for import_path in [source_dir, support_dir]:
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

out_dir.mkdir(parents=True, exist_ok=True)
provider_env = Path(os.getenv("TRAJ_EVAL_PROVIDER_ENV", repo_root / "configs" / "qwen.remote.local.env")).expanduser()
loaded_env = load_env_file(provider_env) if provider_env.exists() else {}
base_url = (os.getenv("OPENAI_BASE_URL") or os.getenv("OPENAI_API_BASE") or "").strip()
model = (os.getenv("CMBAGENT_EVAL_LOCAL_MODEL") or os.getenv("TRAJ_EVAL_MODEL") or os.getenv("OPENAI_MODEL") or "").strip()

qwen_request_timeout = float(os.getenv("QWEN_REQUEST_TIMEOUT", "600"))
qwen_max_retries = int(os.getenv("QWEN_MAX_RETRIES", "10"))
max_iterations = int(os.getenv("STARGAZER_AGENT_MAX_ITERATIONS", "5"))

setup_table = pd.DataFrame([
    ["repo", str(repo_root)],
    ["task observations", str(obs_path.relative_to(repo_root))],
    ["output directory", str(out_dir.relative_to(repo_root))],
    ["provider env exists", provider_env.exists()],
    ["loaded env keys", ", ".join(sorted(loaded_env)) if loaded_env else "none"],
    ["api key set", bool(os.getenv("OPENAI_API_KEY"))],
    ["model", model or "auto-detect"],
    ["request timeout seconds", qwen_request_timeout],
    ["max retries", qwen_max_retries],
    ["max iterations", max_iterations],
], columns=["field", "value"])
display(setup_table)

assert provider_env.exists(), f"Missing provider env file: {provider_env}"
assert base_url, "OPENAI_BASE_URL or OPENAI_API_BASE is missing."
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is missing."


,field,value
0,repo,C:\Dev\src\github.com\msaeedmt\traj-eval
1,task observations,notebooks\qwen_saeed_stargazer_real1\tasks\sta...
2,output directory,notebooks\qwen_saeed_stargazer_real1\outputs\q...
3,provider env exists,True
4,loaded env keys,"CMBAGENT_EVAL_LOCAL_MODEL, OPENAI_API_BASE, OP..."
5,api key set,True
6,model,openai/Qwen3.5-27B-Q5_K_M.gguf
7,request timeout seconds,600.0
8,max retries,10
9,max iterations,5


## 2. Provider smoke check

The agent notebook performs one model lookup and one short chat call. Longer endpoint experiments belong outside this notebook.


In [2]:
client = OpenAI(
    base_url=base_url,
    api_key=os.getenv("OPENAI_API_KEY"),
    timeout=qwen_request_timeout,
    max_retries=qwen_max_retries,
)
models_response = client.models.list()
available_models = [item.id for item in models_response.data]
if not model and available_models:
    model = available_models[0]
    os.environ["TRAJ_EVAL_MODEL"] = model

probe_response = client.chat.completions.create(
    model=model,
    messages=[{"role": "user", "content": "Reply exactly: agent notebook ready"}],
    temperature=0,
    max_tokens=80,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
probe_text = (probe_response.choices[0].message.content or "").strip()
probe = {
    "provider_reachable": True,
    "model": model,
    "available_model_count": len(available_models),
    "chat_ok": bool(probe_text),
    "finish_reason": probe_response.choices[0].finish_reason,
}
write_json(out_dir / "qwen_probe.json", probe)
display(pd.DataFrame([[k, v] for k, v in probe.items()], columns=["field", "value"]))
assert probe["chat_ok"], "Provider chat probe returned no visible text."


,field,value
0,provider_reachable,True
1,model,openai/Qwen3.5-27B-Q5_K_M.gguf
2,available_model_count,1
3,chat_ok,True
4,finish_reason,stop


## 3. Task preview

The input is a sanitized radial-velocity observation table. The agent sees only observations, not hidden truth.


In [3]:
raw_observations = json.loads(obs_path.read_text(encoding="utf-8"))
if isinstance(raw_observations, dict) and "observations" in raw_observations:
    observation_rows = raw_observations["observations"]
elif isinstance(raw_observations, list):
    observation_rows = raw_observations
else:
    observation_rows = raw_observations.get("data", [])

observations_df = pd.DataFrame(observation_rows)
numeric_cols = [c for c in observations_df.columns if pd.api.types.is_numeric_dtype(observations_df[c])]
time_col = next((c for c in ["times_days", "time_days", "time", "t", "jd"] if c in observations_df.columns), numeric_cols[0])
rv_col = next((c for c in ["rvs_ms", "rv_ms", "rv", "radial_velocity", "velocity"] if c in observations_df.columns), None)
sigma_col = next((c for c in ["sigmas_ms", "sigma_ms", "rv_error", "error", "uncertainty"] if c in observations_df.columns), None)
inst_col = next((c for c in ["instruments", "instrument", "inst", "instrument_id"] if c in observations_df.columns), None)

task_summary = {
    "n_observations": int(len(observations_df)),
    "json_shape": "top-level object with an observations field; observations is a dict of parallel arrays",
    "required_observation_access": "raw = json.load(file); obs = raw['observations']; use obs['times_days'], obs['rvs_ms'], obs['sigmas_ms'], obs['instruments']",
    "columns": list(observations_df.columns),
    "time_column": time_col,
    "rv_column": rv_col,
    "sigma_column": sigma_col,
    "instrument_column": inst_col,
    "baseline_days": float(observations_df[time_col].max() - observations_df[time_col].min()),
    "instrument_count": int(observations_df[inst_col].nunique()) if inst_col else None,
}
write_json(out_dir / "task_summary.json", task_summary)
display(pd.DataFrame([[k, v] for k, v in task_summary.items()], columns=["field", "value"]))
display(observations_df.head())



,field,value
0,n_observations,639
1,json_shape,top-level object with an observations field; o...
2,required_observation_access,raw = json.load(file); obs = raw['observations...
3,columns,"[instruments, rvs_ms, sigmas_ms, times_days]"
4,time_column,times_days
5,rv_column,rvs_ms
6,sigma_column,sigmas_ms
7,instrument_column,instruments
8,baseline_days,7237.393404
9,instrument_count,6


,instruments,rvs_ms,sigmas_ms,times_days
0,inst_A,-6.0,9.0,2.449611e+06
1,inst_A,27.0,9.0,2.449612e+06
2,inst_A,-20.0,7.0,2.449655e+06
3,inst_A,-58.0,7.0,2.449728e+06
4,inst_A,4.0,7.0,2.449729e+06


## 4. Agent roles and output contract

The role names match the repo schema: planner, engineer, critic, executor. The engineer must write a standalone script that reads `stargazer_observations.json` and writes `agent_submission.json`.


In [4]:
from traj_eval.agents.roles import (
    CRITIC_SYSTEM_MESSAGE,
    ENGINEER_SYSTEM_MESSAGE,
    PLANNER_SYSTEM_MESSAGE,
)
from stargazer.evaluator import _parse_submission_planets

agent_contract = {
    "input_file": "stargazer_observations.json",
    "output_file": "agent_submission.json",
    "hidden_truth_available_to_agents": False,
    "engineer_rules": [
        "return Python code only",
        "read stargazer_observations.json from the working directory",
        "parse the top-level observations field before reading times_days/rvs_ms/sigmas_ms/instruments",
        "write agent_submission.json in the working directory",
        "do not read hidden truth, benchmark, answer, or hidden task files",
    ],
}
critic_criteria = [
    "Code is standalone and runnable from the workflow directory.",
    "Code uses only public observations and repo-local public support logic.",
    "Code correctly handles the top-level observations object.",
    "Code writes agent_submission.json with a planets list.",
    "Planet fields are numeric and physically plausible: P_days > 0, m_sin_i_mjup >= 0, 0 <= e < 1.",
    "No hidden benchmark, truth file, answer file, or real_001 hidden task file is read.",
    "Execution succeeds and the produced JSON is ready for the separate evaluation notebook.",
]
agent_contract["critic_acceptance_criteria"] = critic_criteria
write_json(out_dir / "agent_workflow_contract.json", agent_contract)
display(pd.DataFrame([[k, v] for k, v in agent_contract.items()], columns=["field", "value"]))
display(pd.DataFrame({"critic_criteria": critic_criteria}))


,field,value
0,input_file,stargazer_observations.json
1,output_file,agent_submission.json
2,hidden_truth_available_to_agents,False
3,engineer_rules,"[return Python code only, read stargazer_obser..."
4,critic_acceptance_criteria,[Code is standalone and runnable from the work...


,critic_criteria
0,Code is standalone and runnable from the workf...
1,Code uses only public observations and repo-lo...
2,Code correctly handles the top-level observati...
3,Code writes agent_submission.json with a plane...
4,Planet fields are numeric and physically plaus...
5,"No hidden benchmark, truth file, answer file, ..."
6,Execution succeeds and the produced JSON is re...


## 5. Qwen call helpers

The helpers are intentionally small: extract visible text, call one role, parse reviewer verdicts, and run the approved script.


In [5]:
qwen_trace = []
transition_records = []
code_review_records = []
result_review_records = []
executor_records = []
submission_validation_records = []


def extract_qwen_text(message) -> str:
    content = (getattr(message, "content", None) or "").strip()
    reasoning = (getattr(message, "reasoning_content", None) or "").strip()
    return content or reasoning


def qwen_role_call(role: str, system_message: str, user_message: str, *, phase: str, iteration: int, max_tokens: int = 1400, enable_thinking: bool = True) -> dict:
    start = time.time()
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_message}, {"role": "user", "content": user_message}],
        temperature=0,
        max_tokens=max_tokens,
        extra_body={"chat_template_kwargs": {"enable_thinking": enable_thinking}},
    )
    message = response.choices[0].message
    record = {
        "role": role,
        "phase": phase,
        "iteration": iteration,
        "text": extract_qwen_text(message),
        "finish_reason": response.choices[0].finish_reason,
        "elapsed_seconds": round(time.time() - start, 3),
        "reasoning_present": bool(getattr(message, "reasoning_content", None)),
        "tokens": response.usage.model_dump() if response.usage else {},
    }
    qwen_trace.append(record)
    return record


def strip_code_fences(text: str) -> str:
    lines = text.strip().splitlines()
    if lines and lines[0].strip().startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].strip().startswith("```"):
        lines = lines[:-1]
    return "\n".join(lines).strip()


def parse_verdict(text: str, approve: str, revise: str) -> str:
    upper = text.upper()
    if approve in upper:
        return approve
    if revise in upper:
        return revise
    return revise


def static_code_findings(code: str) -> list[str]:
    findings = []
    lower = code.lower()
    try:
        ast.parse(code)
    except SyntaxError as exc:
        findings.append(f"syntax error: line {exc.lineno}, {exc.msg}")
    for required in ["stargazer_observations.json", "agent_submission.json", "observations"]:
        if required not in code:
            findings.append(f"missing required JSON handling: {required}")
    for unsafe in ["subprocess", "os.system", "eval(", "exec(", "socket", "requests", "urllib", "openai"]:
        if unsafe in lower:
            findings.append(f"unnecessary capability: {unsafe}")
    for forbidden in ["hidden_task_file", "truth_planets", "answer.json"]:
        if forbidden in lower:
            findings.append(f"forbidden hidden-reference pattern: {forbidden}")
    return findings


def validate_submission_artifact(submission, n_observations: int) -> tuple[bool, list[dict]]:
    rows = []

    def add(check, passed, detail):
        rows.append({"check": check, "passed": bool(passed), "detail": detail})

    add("submission is object", isinstance(submission, dict), type(submission).__name__)
    if not isinstance(submission, dict):
        return False, rows

    planets = submission.get("planets")
    add("planets is list", isinstance(planets, list), f"type={type(planets).__name__}")
    if not isinstance(planets, list):
        return False, rows

    try:
        _parse_submission_planets(submission, "params_only")
        add("repo parser accepts planets", True, "stargazer.evaluator._parse_submission_planets")
    except Exception as exc:
        add("repo parser accepts planets", False, repr(exc))

    required_fields = ["P_days", "m_sin_i_mjup", "e", "omega_rad", "l_rad"]
    for idx, planet in enumerate(planets):
        add(f"planet {idx} is object", isinstance(planet, dict), type(planet).__name__)
        if not isinstance(planet, dict):
            continue
        for field in required_fields:
            value = planet.get(field)
            is_number = isinstance(value, (int, float)) and math.isfinite(float(value))
            add(f"planet {idx} {field} numeric", is_number, value)
        if isinstance(planet.get("P_days"), (int, float)):
            add(f"planet {idx} P_days positive", float(planet["P_days"]) > 0, planet["P_days"])
        if isinstance(planet.get("m_sin_i_mjup"), (int, float)):
            add(f"planet {idx} mass non-negative", float(planet["m_sin_i_mjup"]) >= 0, planet["m_sin_i_mjup"])
        if isinstance(planet.get("e"), (int, float)):
            add(f"planet {idx} eccentricity range", 0 <= float(planet["e"]) < 1, planet["e"])

    rv_model = submission.get("rv_model")
    if rv_model is not None:
        add("rv_model length matches observations", isinstance(rv_model, list) and len(rv_model) == n_observations, f"len={len(rv_model) if isinstance(rv_model, list) else 'not-list'}")
    noise = submission.get("noise", {})
    if isinstance(noise, dict) and "sigma_jitter_ms" in noise:
        value = noise["sigma_jitter_ms"]
        add("noise sigma_jitter_ms non-negative", isinstance(value, (int, float)) and float(value) >= 0, value)

    return all(row["passed"] for row in rows), rows


def run_engineer_script(script_path: Path, cwd: Path) -> dict:
    result = run_command([sys.executable, str(script_path)], cwd=cwd, timeout=int(qwen_request_timeout))
    submission_path = cwd / "agent_submission.json"
    parsed_submission = None
    parse_error = None
    if submission_path.exists():
        try:
            parsed_submission = json.loads(submission_path.read_text(encoding="utf-8"))
        except Exception as exc:
            parse_error = repr(exc)
    return {
        "role": "executor",
        "script_path": str(script_path),
        "cwd": str(cwd),
        "exit_code": result["returncode"],
        "elapsed_seconds": result["elapsed_seconds"],
        "stdout": result["stdout"],
        "stderr": result["stderr"],
        "submission_path": str(submission_path),
        "parsed_submission": parsed_submission,
        "parse_error": parse_error,
    }


def save_run_logs() -> None:
    write_json(out_dir / "qwen_trace_full.json", qwen_trace)
    write_json(out_dir / "agent_transition_trace.json", transition_records)
    write_json(out_dir / "code_review_decisions.json", code_review_records)
    write_json(out_dir / "result_review_decisions.json", result_review_records)
    write_json(out_dir / "executor_records.json", executor_records)
    write_json(out_dir / "submission_validation_records.json", submission_validation_records)


## 6. Planner

The planner only writes the experiment plan. It does not solve the task or invent planet parameters.


In [6]:
planner_prompt = f"""
Plan a real Stargazer radial-velocity finding task and define the critic acceptance criteria.

Observation summary:
{json.dumps(task_summary, indent=2)}

Engineer output contract:
{json.dumps(agent_contract, indent=2)}

Write two short sections:
1. Engineer plan: at most five concrete steps.
2. Critic criteria: how the critic should accept or reject code and final JSON using only public evidence.

Do not provide final planet parameters.
"""
planner_record = qwen_role_call(
    "planner",
    PLANNER_SYSTEM_MESSAGE + "\nPlan only. Define critic criteria. Do not solve the hidden scientific task.",
    planner_prompt,
    phase="planning",
    iteration=0,
    max_tokens=12000,
)
planner_summary = planner_record["text"].strip()
write_json(out_dir / "planner_record.json", planner_record)
(out_dir / "planner_summary.txt").write_text(planner_summary, encoding="utf-8")
transition_records.append({"role": "planner", "phase": "planning", "iteration": 0, "verdict": "PLAN_WRITTEN", "elapsed_seconds": planner_record["elapsed_seconds"], "details": planner_summary[:240]})
save_run_logs()
display(Markdown(planner_summary))


### 1. Engineer Plan

1. Load `stargazer_observations.json` and parse the top-level `observations` dictionary to access parallel arrays.
2. Preprocess the `times_days`, `rvs_ms`, `sigmas_ms`, and `instruments` columns, handling any instrument-specific offsets if necessary.
3. Execute a radial-velocity analysis algorithm to detect periodic signals and fit Keplerian parameters.
4. Format detected signals into a list of planet dictionaries with keys `P_days`, `m_sin_i_mjup`, and `e`.
5. Write the planet list to `agent_submission.json` and verify the file is valid JSON before termination.

### 2. Critic Criteria

1. Code must be standalone Python that reads only `stargazer_observations.json` from the working directory.
2. Code must correctly index `raw['observations']` before accessing `times_days`, `rvs_ms`, `sigmas_ms`, or `instruments`.
3. Output file `agent_submission.json` must exist and contain a top-level `planets` list.
4. Every planet entry must have numeric fields satisfying `P_days > 0`, `m_sin_i_mjup >= 0`, and `0 <= e < 1`.
5. Execution must succeed without attempting to read hidden truth, benchmark, or answer files.

## 7. Engineer, reviewer, executor loop

Each row in the table is one visible workflow step. The reviewer first checks code. The executor only runs reviewer-approved code. The reviewer then checks the result.


In [7]:
if work_dir.exists():
    shutil.rmtree(work_dir)
work_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(obs_path, work_dir / "stargazer_observations.json")

feedback = ""
final_submission_path = None
final_submission = None
engineer_code = ""
loop_stop_reason = "max_iterations"
ready_for_evaluation = False
latest_validation_rows = []

for iteration in range(1, max_iterations + 1):
    engineer_prompt = f"""
Planner plan and critic criteria:
{planner_summary}

Observation summary:
{json.dumps(task_summary, indent=2)}

Contract:
{json.dumps(agent_contract, indent=2)}

Previous critic/executor feedback:
{feedback or 'none'}

Write a standalone Python script. Return code only.
"""
    engineer_record = qwen_role_call(
        "engineer",
        ENGINEER_SYSTEM_MESSAGE + "\nReturn Python code only, without markdown fences.",
        engineer_prompt,
        phase="write_code",
        iteration=iteration,
        max_tokens=42000,
    )
    engineer_code = strip_code_fences(engineer_record["text"])
    script_path = work_dir / f"engineer_iteration_{iteration:02d}.py"
    script_path.write_text(engineer_code, encoding="utf-8")

    findings = static_code_findings(engineer_code)
    code_review_prompt = f"""
Review this engineer script for the public Stargazer contract.

Critic criteria:
{json.dumps(critic_criteria, indent=2)}

Static findings from notebook checks:
{json.dumps(findings, indent=2)}

Script:
```python
{engineer_code[:14000]}
```

Return this exact compact format and finish with END_REVIEW:
VERDICT: APPROVE_CODE or REVISE_CODE
BLOCKING_REASON: one concrete paragraph, maximum 120 words
EVIDENCE:
- exact code behavior or missing requirement
- exact consequence for the Stargazer contract
REPAIR:
- one actionable instruction for the engineer
END_REVIEW

Do not restate every criterion. Do not include hidden reasoning. Do not analyze non-blocking style issues.
"""
    code_review = qwen_role_call(
        "critic",
        CRITIC_SYSTEM_MESSAGE + "\nYou are in code_review mode. Judge contract compliance only; do not use hidden truth.",
        code_review_prompt,
        phase="code_review",
        iteration=iteration,
        max_tokens=2200,
        enable_thinking=False,
    )
    if code_review.get("finish_reason") == "length" and "END_REVIEW" not in code_review.get("text", ""):
        code_review["text"] += "\n\n[Notebook note: critic output hit the token limit before END_REVIEW.]"
    code_verdict = parse_verdict(code_review["text"], "APPROVE_CODE", "REVISE_CODE")
    if findings:
        code_verdict = "REVISE_CODE"
        feedback = "Static code findings: " + "; ".join(findings)
    else:
        feedback = code_review["text"][:1400]
    code_review_records.append({**code_review, "verdict": code_verdict, "static_findings": findings})
    transition_records.append({"role": "engineer", "phase": "write_code", "iteration": iteration, "verdict": "CODE_WRITTEN", "elapsed_seconds": engineer_record["elapsed_seconds"], "details": str(script_path.relative_to(notebook_dir))})
    transition_records.append({"role": "critic", "phase": "code_review", "iteration": iteration, "verdict": code_verdict, "elapsed_seconds": code_review["elapsed_seconds"], "details": feedback[:240]})
    save_run_logs()

    if code_verdict != "APPROVE_CODE":
        continue

    executor_record = run_engineer_script(script_path, work_dir)
    executor_records.append(executor_record)
    final_submission_path = Path(executor_record["submission_path"])
    final_submission = executor_record["parsed_submission"]
    transition_records.append({"role": "executor", "phase": "execute", "iteration": iteration, "verdict": "EXECUTION_OK" if executor_record["exit_code"] == 0 else "EXECUTION_FAIL", "elapsed_seconds": executor_record["elapsed_seconds"], "details": executor_record["stderr"][:240] or executor_record["stdout"][:240]})

    schema_ok, latest_validation_rows = validate_submission_artifact(final_submission, len(observations_df))
    submission_validation_records.append({"iteration": iteration, "schema_ok": schema_ok, "rows": latest_validation_rows})
    validation_preview = pd.DataFrame(latest_validation_rows).to_string(index=False) if latest_validation_rows else "no validation rows"

    result_review_prompt = f"""
Review the executed result. Approve only if the engineer output is ready for the separate evaluation notebook.
Do not use hidden truth or benchmark evidence.

Critic criteria:
{json.dumps(critic_criteria, indent=2)}

Exit code: {executor_record['exit_code']}
Parse error: {executor_record['parse_error']}
Schema validation passed: {schema_ok}
Schema validation rows:
{validation_preview[:5000]}

Submission preview:
{json.dumps(final_submission, indent=2)[:5000]}

Return this exact compact format and finish with END_REVIEW:
VERDICT: APPROVE_RESULT or REVISE_RESULT
BLOCKING_REASON: one concrete paragraph, maximum 120 words
EVIDENCE:
- exact execution or JSON/schema evidence
- exact consequence for evaluation readiness
REPAIR:
- one actionable instruction for the engineer
END_REVIEW

Do not include hidden scientific judgment. Do not include hidden reasoning.
"""
    result_review = qwen_role_call(
        "critic",
        CRITIC_SYSTEM_MESSAGE + "\nYou are in result_review mode. Judge execution and JSON readiness only; do not judge hidden scientific correctness.",
        result_review_prompt,
        phase="result_review",
        iteration=iteration,
        max_tokens=1800,
        enable_thinking=False,
    )
    if result_review.get("finish_reason") == "length" and "END_REVIEW" not in result_review.get("text", ""):
        result_review["text"] += "\n\n[Notebook note: critic output hit the token limit before END_REVIEW.]"
    result_verdict = parse_verdict(result_review["text"], "APPROVE_RESULT", "REVISE_RESULT")
    if executor_record["exit_code"] != 0 or final_submission is None or not schema_ok:
        result_verdict = "REVISE_RESULT"
    result_review_records.append({**result_review, "verdict": result_verdict, "schema_ok": schema_ok})
    transition_records.append({"role": "critic", "phase": "result_review", "iteration": iteration, "verdict": result_verdict, "elapsed_seconds": result_review["elapsed_seconds"], "details": result_review["text"][:240]})
    save_run_logs()

    if result_verdict == "APPROVE_RESULT":
        loop_stop_reason = "APPROVE_RESULT"
        ready_for_evaluation = True
        break
    feedback = result_review["text"][:1400]

iteration_summary = pd.DataFrame(transition_records)
display(iteration_summary[["role", "phase", "iteration", "verdict", "elapsed_seconds", "details"]])
assert transition_records, "The agent loop produced no visible transitions."


,role,phase,iteration,verdict,elapsed_seconds,details
0,planner,planning,0,PLAN_WRITTEN,68.738,### 1. Engineer Plan\n\n1. Load `stargazer_obs...
1,engineer,write_code,1,CODE_WRITTEN,165.139,outputs\qwen_saeed_agent_stargazer\agent_workf...
2,critic,code_review,1,REVISE_CODE,11.992,VERDICT: REVISE_CODE\nBLOCKING_REASON: The scr...
3,engineer,write_code,2,CODE_WRITTEN,120.750,outputs\qwen_saeed_agent_stargazer\agent_workf...
4,critic,code_review,2,REVISE_CODE,9.614,"Static code findings: syntax error: line 105, ..."
5,engineer,write_code,3,CODE_WRITTEN,109.414,outputs\qwen_saeed_agent_stargazer\agent_workf...
6,critic,code_review,3,REVISE_CODE,13.016,VERDICT: REVISE_CODE\nBLOCKING_REASON: The scr...
7,engineer,write_code,4,CODE_WRITTEN,120.288,outputs\qwen_saeed_agent_stargazer\agent_workf...
8,critic,code_review,4,REVISE_CODE,12.632,VERDICT: REVISE_CODE\nBLOCKING_REASON: The scr...
9,engineer,write_code,5,CODE_WRITTEN,152.863,outputs\qwen_saeed_agent_stargazer\agent_workf...


## 8. Final artifact readiness

This section shows the produced JSON and public schema checks. It does not run the hidden benchmark; scientific evaluation happens in the separate evaluation notebook.


In [8]:
submission_rows = []
if isinstance(final_submission, dict):
    for idx, planet in enumerate(final_submission.get("planets", [])):
        if isinstance(planet, dict):
            submission_rows.append({"planet_index": idx, **planet})
submission_table = pd.DataFrame(submission_rows)
display(submission_table)
write_json(out_dir / "clean_star_finding_results.json", submission_rows)

schema_ok = bool(latest_validation_rows) and all(row["passed"] for row in latest_validation_rows)
submission_schema_rows = pd.DataFrame(latest_validation_rows)
display(submission_schema_rows)

artifact_status = pd.DataFrame([
    ["loop_stop_reason", loop_stop_reason],
    ["ready_for_evaluation", ready_for_evaluation],
    ["submission_path", str(final_submission_path) if final_submission_path else None],
    ["submission_exists", bool(final_submission_path and final_submission_path.exists())],
    ["schema_ok", schema_ok],
    ["submitted_planet_count", len(submission_rows)],
], columns=["field", "value"])
display(artifact_status)
write_json(out_dir / "submission_schema_rows.json", latest_validation_rows)


""


""


,field,value
0,loop_stop_reason,max_iterations
1,ready_for_evaluation,False
2,submission_path,None
3,submission_exists,False
4,schema_ok,False
5,submitted_planet_count,0


## 9. Verdict

The final verdict is about the agent workflow and output readiness. A ready submission can still pass or fail the hidden scientific evaluation later.


In [9]:
final_verdict = {
    "notebook_complete": True,
    "task_type": "real_stargazer_finding",
    "target_star": "real_001",
    "loop_stop_reason": loop_stop_reason,
    "iteration_count": max([r.get("iteration", 0) for r in transition_records] or [0]),
    "code_review_count": len(code_review_records),
    "result_review_count": len(result_review_records),
    "executor_count": len(executor_records),
    "final_submission_path": str(final_submission_path) if final_submission_path else None,
    "submitted_planet_count": len(submission_rows),
    "submission_schema_ok": schema_ok,
    "ready_for_evaluation": ready_for_evaluation,
    "engineer_output_status": "ready_for_evaluation" if ready_for_evaluation else "needs_repair",
    "outputs": str(out_dir),
}
write_json(out_dir / "finding_final_verdict.json", final_verdict)
display(pd.DataFrame([[k, v] for k, v in final_verdict.items()], columns=["field", "value"]))
print(json.dumps(final_verdict, indent=2))
assert final_verdict["notebook_complete"]


,field,value
0,notebook_complete,True
1,task_type,real_stargazer_finding
2,target_star,real_001
3,loop_stop_reason,max_iterations
4,iteration_count,5
5,code_review_count,5
6,result_review_count,0
7,executor_count,0
8,final_submission_path,None
9,submitted_planet_count,0


{
  "notebook_complete": true,
  "task_type": "real_stargazer_finding",
  "target_star": "real_001",
  "loop_stop_reason": "max_iterations",
  "iteration_count": 5,
  "code_review_count": 5,
  "result_review_count": 0,
  "executor_count": 0,
  "final_submission_path": null,
  "submitted_planet_count": 0,
  "submission_schema_ok": false,
  "ready_for_evaluation": false,
  "engineer_output_status": "needs_repair",
  "outputs": "C:\\Dev\\src\\github.com\\msaeedmt\\traj-eval\\notebooks\\qwen_saeed_stargazer_real1\\outputs\\qwen_saeed_agent_stargazer"
}
